In [1]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import cartopy.crs as ccrs
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon, Point
from shapely.ops import transform
import cartopy.feature as cfeature
import matplotlib.patheffects as pe
import textwrap
import matplotlib.colors as colors
from matplotlib.ticker import MaxNLocator
import xarray as xr

In [2]:
# read in tc_basins file so we can filter to a specific ocean basin
polygons_dict = {}

with open("tc_basins.dat", "r") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        parts = line.split(",")
        basin_name = parts[0].replace('"', '')
        n_vertices = int(parts[1])

        lon_vals = list(map(float, parts[2:2+n_vertices]))
        lat_vals = list(map(float, parts[2+n_vertices:2+2*n_vertices]))

        coords = list(zip(lon_vals, lat_vals))
        poly = Polygon(coords)

        if basin_name not in polygons_dict:
            polygons_dict[basin_name] = []
        polygons_dict[basin_name].append(poly)

# Convert to GeoDataFrame
basin_records = []

for name, poly_list in polygons_dict.items():
    if len(poly_list) == 1:
        geom = poly_list[0]
    else:
        geom = MultiPolygon(poly_list)

    basin_records.append({
        "basin name": name,
        "geometry": geom
    })

basins = gpd.GeoDataFrame(basin_records, crs="EPSG:4326")

# fix invalid polygons
basins["geometry"] = basins["geometry"].buffer(0)

# remove empy geometries
basins = basins[~basins.geometry.is_empty]

# read in tc_subbasins_NAtl file
sub_polygons_dict = {}

with open("tc_subbasins_NAtl_v5.dat", "r") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        parts = line.split(",")
        sub_basin_name = parts[0].replace('"', '')
        n_vertices = int(parts[1])

        lon_vals = list(map(float, parts[2:2+n_vertices]))
        lon_vals = [(lon + 180) % 360 - 180 for lon in lon_vals]
        lat_vals = list(map(float, parts[2+n_vertices:2+2*n_vertices]))

        coords = list(zip(lon_vals, lat_vals))
        poly = Polygon(coords)

        if sub_basin_name not in sub_polygons_dict:
            sub_polygons_dict[sub_basin_name] = []
        sub_polygons_dict[sub_basin_name].append(poly)

# Convert to GeoDataFrame
sub_basin_records = []

for name, poly_list in sub_polygons_dict.items():
    if len(poly_list) == 1:
        geom = poly_list[0]
    else:
        geom = MultiPolygon(poly_list)

    sub_basin_records.append({
        "sub_basin_name": name,
        "geometry": geom
    })

sub_basins = gpd.GeoDataFrame(sub_basin_records, crs="EPSG:4326",geometry="geometry")

# fix invalid polygons
sub_basins["geometry"] = sub_basins["geometry"].buffer(0)

# remove empty geometries
sub_basins = sub_basins[~sub_basins.geometry.is_empty]

# longitude conversion
import shapely.ops
def shift_lon(geom):
    return shapely.ops.transform(
        lambda x, y: (((x + 180) % 360) - 180, y),
        geom
    )

# shift lon
sub_basins["geometry"] = sub_basins["geometry"].apply(shift_lon)
basins["geometry"] = basins["geometry"].apply(shift_lon)

In [5]:
# read in net cdf
ds = xr.open_mfdataset("datasets/GPI/GPI_EN_calc/post_landmask/GPI_EN_output_daily_landmasked.nc")

print(ds)

<xarray.Dataset> Size: 114MB
Dimensions:      (time: 6763, lat: 37, lon: 57)
Coordinates:
  * time         (time) datetime64[ns] 54kB 1981-09-01 1981-09-02 ... 2025-10-01
  * lat          (lat) float64 296B 90.0 87.5 85.0 82.5 80.0 ... 7.5 5.0 2.5 0.0
  * lon          (lon) float64 456B -120.0 -117.5 -115.0 ... 15.0 17.5 20.0
    number       int64 8B ...
    expver       (time) <U4 108kB dask.array<chunksize=(6763,), meta=np.ndarray>
    spatial_ref  int64 8B ...
Data variables:
    gpi          (time, lat, lon) float64 114MB dask.array<chunksize=(6763, 37, 57), meta=np.ndarray>


In [7]:
# convert to a dataframe
df = ds['gpi'].to_dataframe(name = 'mean').reset_index()

print(df.head())

# join sub basins
points = gpd.GeoDataFrame(
    df, 
    geometry = gpd.points_from_xy(df.lon, df.lat),
    crs = "EPSG:4326"
)

filtered = gpd.sjoin(
    points,
    basins[basins["basin name"] == "N Atlantic"],
    how = "inner",
    predicate = "within"
)

# join sub basin name for each point
filtered_gdf = gpd.GeoDataFrame(
    filtered,
    geometry=gpd.points_from_xy(
        filtered.lon,
        filtered.lat
    ),
    crs=sub_basins.crs
)

filtered_gdf = filtered_gdf.drop(columns="index_right", errors="ignore")

filtered_join = gpd.sjoin(
    filtered_gdf,
    sub_basins[['sub_basin_name', 'geometry']],
    how='left',
    predicate='within'
)


        time   lat    lon  number expver  spatial_ref  mean
0 1981-09-01  90.0 -120.0       0   0001            0   NaN
1 1981-09-01  90.0 -117.5       0   0001            0   NaN
2 1981-09-01  90.0 -115.0       0   0001            0   NaN
3 1981-09-01  90.0 -112.5       0   0001            0   NaN
4 1981-09-01  90.0 -110.0       0   0001            0   NaN


In [8]:
# add year column for group by
filtered_join['year'] = filtered_join['time'].dt.year

annual_table = (
    filtered_join.groupby(["year", "sub_basin_name"])["mean"]
            .mean()
            .reset_index()
)

print(annual_table)

# save to csv
annual_table.to_csv("datasets/GPI/GPI_EN_calc/post_landmask/GPI_annual_mean_bySubbasin_table_landmasked.csv")


     year         sub_basin_name      mean
0    1981                 Arctic  0.592659
1    1981              Caribbean  3.729751
2    1981       Central Atlantic  1.023886
3    1981           Deep Tropics  0.828841
4    1981        Eastern Tropics  3.494413
..    ...                    ...       ...
670  2025  Northeastern Seaboard  1.874626
671  2025        Northern Europe  0.475165
672  2025  Southeastern Seaboard  6.230220
673  2025   Subtropical Atlantic  0.228024
674  2025         Western Africa       NaN

[675 rows x 3 columns]
